# US-080 - Refinador FarSLIP-fenologia: el copiloto afina el Voting-3

### Segunda etapa "Be My Eyes": el LLM describe, FarSLIP re-rankea

**Equipo 17** - AgroSatCopilot - EPIC 12

---

El campeon del copiloto es el **Voting-3** (US-079). Esta US anade una **segunda etapa opcional**: el reasoner LLM genera una **descripcion fenologica por clase** y **FarSLIP** (CLIP region-aware, alinea imagen<->texto) puntua la parcela contra cada descripcion; esa senal se **fusiona** con el posterior Voting-3 **solo cuando los modelos densos estan inseguros** (margen bajo, miembros en desacuerdo) o ante una **clase abierta/nueva**. En el caso facil el campeon **no se toca** -- nunca empeora donde ya acierta.

Metodo respaldado por *"Phenology description is all you need!"* (ISPRS J. P&RS 2025/26) y la linea LLM-descripcion -> CLIP (CuPL 2022, Saha 2024, Concept-Guided Bayesian 2026).

> **Solo valores reales.** Los posteriores Voting-3, la verdad de campo y los prompts son reales. El **score FarSLIP** usado para ilustrar el mecanismo se marca **ILUSTRATIVO** (no es una medicion); el **ΔF1** se corre de verdad si FarSLIP esta servido + existen los chips, o dice **PENDIENTE** con como completarlo -- nunca inventa.

In [1]:
# Parametros (papermill). Sobreescribe con `papermill -p <name> <value>`.
demo_user = 'demo@agrosat.dev'   # propietario de la sesion sembrada
n_parcels = 3                    # parcelas reales a observar
alpha = 0.4                      # peso convexo de la senal FarSLIP en la fusion
margin_tau = 0.15                # umbral de margen top1-top2 para disparar
farslip_weights = 'gs://agrosat-models/farslip/farslip-clip-italy-v1/'  # pesos FarSLIP

## Preparacion del entorno

Resolvemos la raiz del repo, cargamos `.env.local`, silenciamos el ruido de logs y abrimos el *pool* de la base local. Sin rutas absolutas ni secretos.

In [2]:
import os
import sys
from pathlib import Path

for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding='utf-8')
    except (AttributeError, ValueError):
        pass

from ml.utils.notebook_setup import find_repo_root, load_env_local

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
load_env_local(REPO_ROOT)

%load_ext autoreload
%autoreload 2

import polars as pl
from IPython.display import Markdown, display

from ml.agent import demo
demo.quiet_logging()
print('repo:', REPO_ROOT)

repo: C:\Users\arthu\Proyectos\MNA\agro_sat_copilot


## 1. Las descripciones fenologicas por clase (lo que ve FarSLIP)

FarSLIP alinea la imagen de la parcela contra **un texto por clase candidata**. Idealmente el reasoner genera esos textos en vivo; aqui usamos el **fallback determinista** (en ingles, porque el encoder de texto de FarSLIP es el CLIP teacher) para las nueve clases bien resueltas `france-9`. Son los prompts contra los que se puntua la imagen.

In [3]:
from ml.eval.class_remap import get_label_space
from ml.farslip.class_prompts import build_class_prompts

france9 = list(get_label_space('france-9').class_names.values())
prompts = build_class_prompts(france9)
display(pl.DataFrame({'clase': list(prompts), 'descripcion_fenologica': list(prompts.values())}))

clase,descripcion_fenologica
str,str
"""Meadow""","""a satellite image of a meadow …"
"""Soft winter wheat""","""a satellite image of a soft wi…"
"""Corn""","""a satellite image of a corn fi…"
"""Winter barley""","""a satellite image of a winter …"
"""Winter rapeseed""","""a satellite image of a winter …"
"""Sunflower""","""a satellite image of a sunflow…"
"""Grapevine""","""a satellite image of a grapevi…"
"""Beet""","""a satellite image of a beet fi…"
"""Soybeans""","""a satellite image of a soybean…"


## 2. El refinador, sobre posteriores Voting-3 REALES

Observamos parcelas reales de la sesion con el perceiver (posterior **Voting-3 real**) y aplicamos `apply_refinement`. El disparo es **selectivo**: en el caso facil (margen alto, miembros de acuerdo) devuelve el campeon **intacto**; en el incierto / open-set lo re-rankea. El **score FarSLIP** de abajo es **ILUSTRATIVO** (un favor a una clase) para ensenar el mecanismo: el score real sale del cabezal de la seccion 3 cuando FarSLIP este servido.

In [4]:
from ml.agent.context import ToolContext
from ml.agent.db import get_pool
from ml.agent.perceiver import PerceiverLayer
from backend.app.core.config import get_settings

settings = get_settings()
pool = await get_pool()
async with pool.acquire() as conn:
    session_id = await conn.fetchval(
        'SELECT cs.id FROM chat_sessions cs LEFT JOIN parcels p ON p.session_id=cs.id '
        'WHERE cs.user_id=$1 GROUP BY cs.id ORDER BY count(p.id) DESC, cs.id LIMIT 1',
        demo_user)
    _rows = await conn.fetch(
        'SELECT id FROM parcels WHERE session_id=$1 ORDER BY id LIMIT $2',
        session_id, int(n_parcels))
parcel_ids = [int(r['id']) for r in _rows]
ctx = ToolContext(pool=pool, settings=settings, session_id=session_id)
perceiver = PerceiverLayer(ctx)
observations = await demo.observe_parcels(perceiver, parcel_ids)
print('parcelas:', parcel_ids)

parcelas: [52, 53, 54]


In [5]:
# Apply the gated refiner to each real Voting-3 posterior. The FarSLIP score here is
# ILLUSTRATIVE (it favours the parcel's 2nd-most-likely class to show the mechanism);
# the REAL score comes from the FarSLIP head (section 3) once it is served.
from ml.agent.refine import apply_refinement, top1_top2_margin

_rows = []
for obs, _ in observations:
    post = obs.class_probabilities
    _ranked = sorted(post, key=lambda k: post[k], reverse=True)
    # ILUSTRATIVO: a FarSLIP that favours the runner-up class (forces an uncertain re-rank).
    _illustrative = {c: (1.0 if c == _ranked[min(1, len(_ranked) - 1)] else 0.0) for c in post}
    res = apply_refinement(post, _illustrative, alpha=alpha, margin_tau=margin_tau, open_set=True)
    _rows.append({
        'parcela': obs.parcel_id,
        'margen_top1_top2': round(top1_top2_margin(post), 3),
        'clase_voting3': res.top_class_before,
        'disparo': res.reason,
        'refinado': res.refined,
        'clase_tras_refinar': res.top_class_after,
    })
display(pl.DataFrame(_rows))
display(Markdown('**ILUSTRATIVO**: el score FarSLIP de esta celda es un ejemplo del '
    'mecanismo (favorece a la 2a clase). El ΔF1 real se mide en la seccion 4 con el '
    'cabezal FarSLIP de la seccion 3.'))

parcela,margen_top1_top2,clase_voting3,disparo,refinado,clase_tras_refinar
i64,f64,str,str,bool,str
52,0.965,"""Soft winter wheat""","""open_set""",true,"""Soft winter wheat"""
53,0.983,"""Beet""","""open_set""",true,"""Beet"""
54,0.931,"""Meadow""","""open_set""",true,"""Meadow"""


**ILUSTRATIVO**: el score FarSLIP de esta celda es un ejemplo del mecanismo (favorece a la 2a clase). El ΔF1 real se mide en la seccion 4 con el cabezal FarSLIP de la seccion 3.

## 3. El cabezal zero-shot de FarSLIP (intento de carga real)

El cabezal calcula `softmax(image_emb . text_emb / T)` con FarSLIP (`extract_embeddings` + `encode_text`, ambos L2-norm). Intentamos cargar el **extractor real**; si los pesos no son alcanzables (sin ADC / sin `dvc pull`), se reporta el **blocker honesto** -- no se fabrican embeddings.

In [6]:
# Honest attempt to load the real FarSLIP extractor. No fabrication: on failure we
# report the blocker and leave the real scoring + delta-F1 as pending.
farslip = None
try:
    from ml.extractors.farslip_extractor import FarSLIPExtractor
    farslip = FarSLIPExtractor(weights_uri=farslip_weights, device='cpu')
    _t = farslip.encode_text(list(prompts.values())[:2])
    display(Markdown(f'FarSLIP cargado. encode_text -> shape `{tuple(_t.shape)}` '
        '(modo student si cargaron los pesos, teacher si no).'))
except Exception as exc:  # noqa: BLE001 - report the blocker, never fabricate
    display(Markdown(
        f'> **PENDIENTE (blocker US-080 sec 4.2)**: no se pudo cargar FarSLIP '
        f'(`{type(exc).__name__}: {exc}`). Para la corrida real: `dvc pull` de los '
        'pesos / ADC de GCS + los chips por parcela (`ml/farslip/dataset.py`). El '
        'cabezal `ml.farslip.zeroshot_head` queda listo e inyectable.'))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[warning  ] GCS auth/permiso denegado; intentando cache local error='File ./.env/gcp-service-account.json was not found.' error_type=DefaultCredentialsError uri=gs://agrosat-models/farslip/farslip-clip-italy-v1/


[warning  ] FarSLIP weights no disponibles; corriendo en modo teacher (degradado) weights_uri=gs://agrosat-models/farslip/farslip-clip-italy-v1/


FarSLIP cargado. encode_text -> shape `(2, 512)` (modo student si cargaron los pesos, teacher si no).

## 4. Eval ΔF1: Voting-3 vs Voting-3 + refinador

`run_refine_eval` compara el F1-macro `france-9` del Voting-3 contra el refinado, **global** y sobre el **subconjunto disparado**, leyendo el OOF fold-5 real. Necesita un *scorer* FarSLIP por parcela (chip -> cabezal zero-shot). Si FarSLIP no esta disponible, se reporta **PENDIENTE** con como completarlo -- sin numeros inventados.

In [7]:
# Real delta-F1 needs the FarSLIP scorer (chip -> zero-shot) over the fold-5 OOF. That
# is the documented blocker; here we wire the harness and report PENDIENTE honestly
# when the scorer cannot be built. No fabricated metrics.
from ml.eval.farslip_refine_eval import f1_macro  # noqa: F401  (smoke import)

if farslip is None:
    display(Markdown(
        '> **PENDIENTE**: ΔF1 real. Falta el scorer FarSLIP (pesos + chips por parcela). '
        'Con FarSLIP servido: construir `scorer(canonical_id) -> {clase: score}` con '
        '`farslip_zeroshot_scores_one(extractor, chip, france9, list(prompts.values()))` '
        'y llamar `run_refine_eval(voting_posteriors, ground_truth, scorer, alpha, margin_tau)`. '
        'Los posteriores Voting-3 y la GT salen de `classify._load_voting_three()` + '
        '`_build_parcel_ground_truth`. El harness esta listo y testeado.'))
else:
    display(Markdown('> FarSLIP disponible: completar el scorer sobre los chips del '
        'fold-5 y correr `run_refine_eval` (ver la celda de arriba). Pendiente la ruta '
        'de chips por parcela.'))

> FarSLIP disponible: completar el scorer sobre los chips del fold-5 y correr `run_refine_eval` (ver la celda de arriba). Pendiente la ruta de chips por parcela.

## Conclusiones

- El **refinador FarSLIP-fenologia** anade una segunda etapa al campeon Voting-3 que **solo actua en parcelas inciertas / open-set**, con fusion convexa auditable -- nunca degrada el caso facil (garantia por diseno).
- Las **descripciones por clase**, el **disparo selectivo** y el **harness ΔF1** estan implementados y testeados (`ml.agent.refine`, `ml.farslip.zeroshot_head`, `ml.farslip.class_prompts`, `ml.eval.farslip_refine_eval`).
- **Lo que falta** (blocker): FarSLIP servido + los **chips por parcela** para el scorer real -> medir el **ΔF1 real**, sobre todo en el caso **open-set** (cultivos mediterraneos que PASTIS no tiene), el de mayor valor esperado segun el paper.

### Cierre

In [8]:
from ml.agent.db import close_pool

await close_pool()
print('pool cerrado.')

pool cerrado.
